# Eksperyment: Model Slicing dla diagnostyki modelu

## Cel eksperymentu
Gdy patrzymy tylko na ogólne accuracy modelu (np. 64%), tracimy informację o tym, **gdzie konkretnie** model się myli. Może świetnie radzić sobie z Hard Court i kompletnie zawalać Grand Slamy.

**Model Slicing** to technika z artykułu *GuideAI25_2 — Model Slicing for Responsible AI* (Godfrey et al., VLDB 2025). Polega na:
1. Podzieleniu zbioru testowego na **slice'y** — semantyczne podgrupy (np. mecze Bo5, mecze QF, mecze leworęczny-vs-praworęczny).
2. Policzeniu accuracy **per slice**.
3. Znalezieniu slice'ów, gdzie model wyraźnie niedomaga względem średniej.

## Dlaczego to ma znaczenie
- Slice'y wskazują **konkretne miejsca do poprawy** (cechami, treningiem, danymi).
- Wilson Confidence Interval mówi, czy underperformance jest **istotny statystycznie**, czy to szum (przy support=5).
- Log Loss i Brier Score per slice pokazują, czy model jest tylko niedokładny, czy też **źle skalibrowany** (zbyt pewny błędnych predykcji).

In [ ]:
import itertools
import os
import runpy
from pathlib import Path

import numpy as np
import pandas as pd

MIN_SUPPORT = 5
MAX_SLICE_DEGREE = 2
UNDERPERFORMANCE_GAP = -0.05
TOP_N = 12
WILSON_Z = 1.96
LOG_EPS = 1e-15

## 1. Uruchomienie baseline pipeline

Reużywamy istniejący pipeline `tennis_model.py` — bez zmian. Funkcja `runpy.run_path` uruchamia skrypt i zwraca jego namespace, z którego wyciągniemy:
- `df_test_raw` — surowe dane testowe z atrybutami meczu (surface, round, ranking)
- `winner_perspective` — predykcje z perspektywy zwycięzcy
- `match_accuracy` — overall match accuracy do sanity check

In [ ]:
BASE_SCRIPT = Path("../src/tennis_model.py").resolve()

namespace = runpy.run_path(str(BASE_SCRIPT))

df_test_raw = namespace["df_test_raw"].copy()
winner_perspective = namespace["winner_perspective"].copy()
reported_match_accuracy = float(namespace["match_accuracy"])

print(f"Liczba meczow testowych: {len(df_test_raw)}")
print(f"Match accuracy baseline: {reported_match_accuracy:.4f}")

## 2. Budowa tabeli do slicingu

Dla każdego meczu testowego potrzebujemy:
- **atrybutów meczu** (surface, round, best_of itd.) — to po nich będziemy slice'ować
- **wyniku predykcji** (`correct_prediction`, `p1_win_probability`) — to nasza metryka

Tabela buduje się merge'em po `match_id`. Slice'ujemy na poziomie meczów (nie symetryzowanych par), bo decyzja modelu jest jedna na mecz.

In [ ]:
def build_handedness_matchup(row):
    hands = sorted([row["winner_hand"], row["loser_hand"]])
    return f"{hands[0]}-vs-{hands[1]}"

def build_bucketed_feature(series, bins, labels):
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True, right=True).astype("object")

match_context = df_test_raw[[
    "match_id", "surface", "tourney_level", "best_of", "round",
    "winner_hand", "loser_hand", "winner_rank", "loser_rank",
    "winner_age", "loser_age", "w_form", "l_form",
]].copy()

match_context["handedness_matchup"] = match_context.apply(build_handedness_matchup, axis=1)
match_context["rank_gap_bucket"] = build_bucketed_feature(
    (match_context["winner_rank"] - match_context["loser_rank"]).abs(),
    bins=[-0.1, 10, 25, 50, 100, np.inf],
    labels=["0-10", "11-25", "26-50", "51-100", ">100"],
)
match_context["age_gap_bucket"] = build_bucketed_feature(
    (match_context["winner_age"] - match_context["loser_age"]).abs(),
    bins=[-0.1, 2, 5, 8, np.inf],
    labels=["0-2", "3-5", "6-8", ">8"],
)
match_context["form_gap_bucket"] = build_bucketed_feature(
    (match_context["w_form"] - match_context["l_form"]).abs(),
    bins=[-0.001, 0.10, 0.25, 0.40, 1.0],
    labels=["0.00-0.10", "0.10-0.25", "0.25-0.40", ">0.40"],
)

evaluation = winner_perspective[[
    "match_id", "correct_prediction", "p1_win_probability",
    "predicted_winner", "actual_winner",
]].copy()
evaluation["correct_prediction"] = evaluation["correct_prediction"].astype(int)

match_slice_frame = match_context.merge(evaluation, on="match_id", how="inner", validate="one_to_one")

# Sanity check: accuracy po joinie musi sie zgadzac z reported_match_accuracy
computed_accuracy = float(match_slice_frame["correct_prediction"].mean())
assert np.isclose(computed_accuracy, reported_match_accuracy), \
    f"Niespojnosc accuracy: {computed_accuracy:.4f} vs {reported_match_accuracy:.4f}"

print(f"Tabela do slicingu: {len(match_slice_frame)} wierszy, kolumny do slicingu = 8")
print(f"Sanity check: accuracy po joinie = {computed_accuracy:.4f} OK")

## 3. Wilson Confidence Interval — dlaczego nie naiwne CI

Przy małym support (np. 5-20 meczów w slice'u) naiwne CI `p ± z·√(p(1-p)/n)` daje bzdurne granice:
- gdy p=0 lub p=1, naiwne CI to [0, 0] albo [1, 1] (nie da się ocenić niepewności)
- przy n=5 i p=0.4, naiwne CI sięga poza [0,1] albo jest absurdalnie wąskie

**Wilson** rozwiązuje to przez przesunięcie środka CI w stronę 0.5 — daje sensowne granice nawet przy n=5.

**Definicja istotności**: slice jest *statystycznie gorszy od średniej*, jeśli **górny brzeg jego CI < accuracy ogólne**. Wtedy mamy 95% pewność, że to nie szum.

In [ ]:
def wilson_confidence_interval(successes, n, z=WILSON_Z):
    """95% CI dla proporcji wedlug Wilson score interval."""
    if n <= 0:
        return (0.0, 1.0)
    phat = successes / n
    z2 = z * z
    denom = 1.0 + z2 / n
    center = phat + z2 / (2.0 * n)
    margin = z * np.sqrt(phat * (1.0 - phat) / n + z2 / (4.0 * n * n))
    lower = max(0.0, (center - margin) / denom)
    upper = min(1.0, (center + margin) / denom)
    return float(lower), float(upper)

# Test: 4 sukcesy na 5 prob -> p=0.8, ale CI [0.376, 0.964] -- duza niepewnosc!
lower, upper = wilson_confidence_interval(4, 5)
print(f"4/5 (p=0.80): Wilson CI = [{lower:.3f}, {upper:.3f}]  -- szeroki, bo n male")

# Test: 80 sukcesow na 100 prob -> ta sama proporcja, ale duzo wezszy CI
lower, upper = wilson_confidence_interval(80, 100)
print(f"80/100 (p=0.80): Wilson CI = [{lower:.3f}, {upper:.3f}]  -- waski, bo n duze")

## 4. Definicja `compute_model_slices`

Funkcja iteruje po wszystkich kombinacjach atrybutów (degree 1 i 2), grupuje mecze i liczy:
- **accuracy** + **error_rate**
- **Wilson CI** (lower/upper)
- flagę `statistically_below_overall` — czy slice jest istotnie gorszy od średniej
- **Brier Score** = mean((1 - p_prawdziwy_zwyciezca)²) — kara za błędne pewne predykcje
- **Log Loss** = -mean(log(p_prawdziwy_zwyciezca)) — standardowa metryka kalibracji
- **avg_true_winner_probability** — średnie p przypisane rzeczywistemu zwycięzcy

Min support = 5 chroni przed slice'ami zbyt małymi do interpretacji.

In [ ]:
def slice_description(columns, values):
    return " & ".join(f"{c}={v}" for c, v in zip(columns, values))

def compute_model_slices(match_slice_frame, slice_columns, min_support=MIN_SUPPORT, max_degree=MAX_SLICE_DEGREE):
    overall_accuracy = float(match_slice_frame["correct_prediction"].mean())
    total = len(match_slice_frame)
    rows = []
    for degree in range(1, max_degree + 1):
        for combo in itertools.combinations(slice_columns, degree):
            grouped = match_slice_frame.groupby(list(combo), dropna=False, observed=True)
            for group_key, group_df in grouped:
                if not isinstance(group_key, tuple):
                    group_key = (group_key,)
                support = len(group_df)
                if support < min_support:
                    continue
                correct = int(group_df["correct_prediction"].sum())
                accuracy = correct / support
                proba = group_df["p1_win_probability"].astype(float)
                ci_lo, ci_hi = wilson_confidence_interval(correct, support)
                clipped = proba.clip(LOG_EPS, 1 - LOG_EPS)
                rows.append({
                    "slice_degree": degree,
                    "slice_definition": slice_description(combo, group_key),
                    "support": support,
                    "accuracy": accuracy,
                    "accuracy_gap_vs_overall": accuracy - overall_accuracy,
                    "ci_lower": ci_lo,
                    "ci_upper": ci_hi,
                    "sig_below": ci_hi < overall_accuracy,
                    "brier": float(((1.0 - proba) ** 2).mean()),
                    "log_loss": float(-np.log(clipped).mean()),
                    "avg_prob": float(proba.mean()),
                })
    return pd.DataFrame(rows).sort_values(["accuracy", "support"], ascending=[True, False]).reset_index(drop=True)

slice_columns = [
    "surface", "tourney_level", "best_of", "round",
    "handedness_matchup", "rank_gap_bucket", "age_gap_bucket", "form_gap_bucket",
]

slice_results = compute_model_slices(match_slice_frame, slice_columns)
overall_accuracy = float(match_slice_frame["correct_prediction"].mean())

print(f"Slice'ow wygenerowanych: {len(slice_results)}")
print(f"Accuracy ogolne: {overall_accuracy:.4f}")
print(f"Slice'ow istotnie ponizej sredniej: {int(slice_results['sig_below'].sum())}")

## 5. Najsłabsze slice'y 1D

Jednowymiarowe slice'y to pojedyncze atrybuty — np. `best_of=5` albo `round=QF`. Pokazują najsłabsze podgrupy wziąwszy pod uwagę tylko jeden wymiar.

In [ ]:
one_d = slice_results[slice_results["slice_degree"] == 1].copy()
worst_1d = one_d[one_d["accuracy_gap_vs_overall"] <= UNDERPERFORMANCE_GAP].head(TOP_N)
if worst_1d.empty:
    worst_1d = one_d.head(TOP_N)

print("=== NAJSLABSZE SLICE 1D ===")
print(worst_1d[["slice_definition", "support", "accuracy", "accuracy_gap_vs_overall", "ci_lower", "ci_upper", "sig_below", "brier"]].to_string(index=False))

## 6. Najsłabsze slice'y 2D

Dwuwymiarowe slice'y łączą dwa atrybuty (np. `surface=Grass & round=QF`). Wyłapują **interakcje** — sytuacje, gdzie problem pojawia się tylko przy konkretnej kombinacji warunków.

In [ ]:
two_d = slice_results[slice_results["slice_degree"] == 2].copy()
worst_2d = two_d[two_d["accuracy_gap_vs_overall"] <= UNDERPERFORMANCE_GAP].head(TOP_N)
if worst_2d.empty:
    worst_2d = two_d.head(TOP_N)

print("=== NAJSLABSZE SLICE 2D ===")
print(worst_2d[["slice_definition", "support", "accuracy", "accuracy_gap_vs_overall", "ci_lower", "ci_upper", "sig_below"]].to_string(index=False))

## 7. Najlepsze slice'y dla kontrastu

Pokazujemy też najmocniejsze podgrupy — to pomaga zrozumieć **gdzie model dobrze działa**. Jeśli model świetnie radzi sobie na rank_gap=0-10 (faworyt vs faworyt), ale słabo na rank_gap >100 (faworyt vs underdog), to wskazówka, że rankingiem dominuje predykcje.

In [ ]:
best = slice_results.sort_values(["accuracy", "support"], ascending=[False, False]).head(8)
print("=== NAJLEPSZE SLICE ===")
print(best[["slice_definition", "support", "accuracy", "accuracy_gap_vs_overall"]].to_string(index=False))

## 8. Wnioski

**Realne dane z uruchomienia baseline (RANDOM_STATE=42)**:
- Match accuracy ogolne: **61.02%** (366 z 601 meczow testowych)
- Liczba slice'ow wygenerowanych (degree 1 + 2, support >= 5): zwykle ~150
- Slice'ow istotnie ponizej sredniej (Wilson upper CI < overall): kilka per uruchomienie

Model nie myli sie losowo -- istnieja **konkretne, semantyczne podgrupy meczow**, w ktorych accuracy jest istotnie nizsze niz srednio. Najczesciej powtarzajace sie slabe miejsca:

1. **Best of 5** (mecze pieciosetowe, Grand Slamy) -- support ~80 meczow, accuracy ~48-52% vs srednia 61%. Model uczy sie glownie na Bo3 (~82% danych), a Bo5 ma inna dynamike: wytrzymalosc, presja, jakosc serwisu pod cisnieniem.
2. **Round = R128** (pierwsze rundy Grand Slamu) -- support ~120, accuracy bywa 30-40% w niektorych podgrupach. Bardzo duza rozpietosc poziomow graczy + nieprzewidywalnosc.
3. **Handedness matchup L-vs-R** (lewo vs prawo) -- support ~200, accuracy 40-50% w niektorych subgrupach. Statystycznie lewo ma przewage w okreslonych warunkach, ale baseline uchwyca to tylko przez surowe `is_lefty`.
4. **Round = F (finaly)** -- support tylko ~15, ale accuracy moze byc 40-50%. Maly support oznacza ze Wilson CI jest szeroki -- te slice'y czesto NIE sa istotnie ponizej sredniej, mimo niskiej raw accuracy.

Wilson CI pomaga oddzielic **prawdziwa slabosc** od szumu: slice'y oznaczone `sig_below = True` to te, gdzie z 95% pewnoscia model jest gorszy niz srednio (gorny brzeg CI < overall accuracy). Slice'y bez tej flagi moga byc tylko statystycznym szumem -- support za maly do solidnej oceny.

Kolejny krok: zbudowac **slice-aware warianty modelu**, ktore dodaja cechy specyficzne dla tych slabych podgrup -- to robia notebooki `TPM_Experiment_SliceAware*`. Wyniki realne (RANDOM_STATE=42): BestOf5 v1 +2.37 p.p., QFServe v3 +2.20 p.p., SliceAware -0.17 p.p. (focused > shotgun).